## 1. Setup and Imports {#setup}

Let's start by importing all necessary modules and setting up the environment.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import sys
import os
warnings.filterwarnings('ignore')

import sys
import os


from AbXtract import *
from AbXtract import AntibodyDescriptorCalculator, Config, load_config
from AbXtract.sequence import (
    SequenceLiabilityAnalyzer,
    BashourDescriptorCalculator,
    PeptideDescriptorCalculator,
    AntibodyNumbering
)
from AbXtract.structure import (
    SASACalculator,
    ChargeAnalyzer,
    DSSPAnalyzer,
    PropkaAnalyzer,
    ArpeggioAnalyzer
)
from AbXtract.utils import (
    read_fasta,
    write_fasta,
    parse_sequence,
    validate_sequence,
    analysis_descriptors
)


# 2. Function to combine descriptors and class them

In [2]:
def desc_Ab(HEAVY_SEQUENCE, LIGHT_SEQUENCE, PDB_FILE):
    
    if HEAVY_SEQUENCE:
        heavy_valid, heavy_msg = validate_sequence(HEAVY_SEQUENCE)
        heavy_numbered = numbering.number_sequence(HEAVY_SEQUENCE, 'H')  # Use VH portion only
        annotated_H, cdrs_H = numbering.get_cdr_sequences(heavy_numbered, 'H')
        heavy_profiles = numbering.get_peptide_profiles(HEAVY_SEQUENCE)
    
    if LIGHT_SEQUENCE:
        light_valid, light_msg = validate_sequence(LIGHT_SEQUENCE)
        light_numbered = numbering.number_sequence(LIGHT_SEQUENCE, 'L')  # Use VH portion only    
        annotated_L, cdrs_L = numbering.get_cdr_sequences(light_numbered, 'L')
        light_profiles = numbering.get_peptide_profiles(LIGHT_SEQUENCE)

    peptide_results = peptide_calc.calculate_all(
    heavy_sequence=HEAVY_SEQUENCE,
    light_sequence=LIGHT_SEQUENCE
    )

    sequence_results, liabilities = calc.calculate_sequence_descriptors(
    heavy_sequence=HEAVY_SEQUENCE,
    light_sequence=LIGHT_SEQUENCE,
    sequence_id="TestAb_Sequence"
    )

    structure_results_seq, structure_results_comp, df_residues, df_AA, df_Ab = calc.calculate_structure_descriptors(
    heavy_sequence=HEAVY_SEQUENCE,
    light_sequence=LIGHT_SEQUENCE,
    pdb_file=PDB_FILE,
    structure_id="TestAb_Structure"
    )
    
        
    # Ensure residue_sasa_sum column exists
    structures_results_seq = analysis_descriptors.add_residue_sasa_sum_column(structure_results_seq)

    # Get data
    liabilities_list = liabilities['liabilities'].iloc[0]
    structures_data = structures_results_seq.iloc[0]

    if HEAVY_SEQUENCE:
        df_heavy_final = analysis_descriptors.create_complete_antibody_dataframe( 0, df_residues, df_Ab, 
            HEAVY_SEQUENCE, annotated_H, heavy_profiles, 
            structures_data, liabilities_list, 'Heavy', 'imgt'
                                                           
        )
    else:
        df_light_final = None
        
    if LIGHT_SEQUENCE:
        df_light_final = analysis_descriptors.create_complete_antibody_dataframe(len(HEAVY_SEQUENCE), df_residues, df_Ab, 
            LIGHT_SEQUENCE, annotated_L, light_profiles,
            structures_data, liabilities_list, 'Light', 'imgt'
                                                            
        )
    else:
        df_light_final = None
        
    df_final = analysis_descriptors.combine_all_results(
        df_AA,
        structure_results_comp,
        sequence_results,
        peptide_results,
        heavy_valid=heavy_valid,
        light_valid=light_valid,
        cdrs_H=cdrs_H,
        cdrs_L=cdrs_L
    )

    return(df_heavy_final, df_light_final, df_final)

# Load config

In [3]:
# default configuration
custom_config = Config()

'''
# Test custom configuration
custom_config = Config.from_dict({
    'pH': 7.4,
    'numbering_scheme': 'kabat',
    'verbose': True,
    'calculate_dssp': tool_status.get('dssp', False),
    'calculate_propka': tool_status.get('propka', False),
    'calculate_arpeggio': tool_status.get('arpeggio', False)
})
'''


# Check external tool availability
tool_status = custom_config.check_external_tools()
print("🛠️ External Tool Status:")
for tool, available in tool_status.items():
    status = "OK" if available else "Fail"
    print(f"  {tool}: {status}")


reduce not found at reduce


🛠️ External Tool Status:
  dssp: OK
  propka: OK
  arpeggio: OK
  reduce: Fail
  muscle: OK


# Load classes

In [4]:
numbering = AntibodyNumbering(scheme='imgt')
peptide_calc = PeptideDescriptorCalculator()
calc = AntibodyDescriptorCalculator(config=custom_config)

reduce not found at reduce
Missing external tools: reduce
Some analyses may not be available


--- 0.48187708854675293 seconds --- INITIALIZATION


# Define path 

In [5]:
abxtract_path = "/home/HX46_FR5/repo_perso/AbXtract"
sys.path.insert(0, abxtract_path)

# Set up test data paths
BASE_DIR = Path.cwd() 
DATA_DIR = BASE_DIR / "data" / "test"
DATA_DIR.mkdir(parents=True, exist_ok=True)


# Define test file paths
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)


# Input sequence and pdb

In [6]:
# Test antibody sequences (based on therapeutic antibodies)
HEAVY_SEQUENCE_list = (
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGRYGIHWVRQAPGKGLEWMGGISPSGGTTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGRYGIHWVRQAPGKGLEWMGGINPSGYGTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGSSAIHWVRQAPGKGLEWMGGISPSFGTAIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTLSRYGIHWVRQAPGKGLEWMGGISPSFGTAIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGRYGIHWVRQAPGKGLEWMGGISPSGGTTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFSRAAIHWVRQAPGKGLEWMGGSIPMFGTTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGRYGIHWVRQAPGKGLEWMGGISPSGGTTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS"
)
# Light chain: Includes realistic VL domain + human kappa constant region  
LIGHT_SEQUENCE_list = (
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLHSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASQDIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASEDIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK"
)
PDB_FILE_list = [
    DATA_DIR / "AAS-41124-[41032.41124].pdb",
    DATA_DIR / "AAS-41125-[41033.41125].pdb",
    DATA_DIR / "AAS-41126-[41034.41126].pdb",
    DATA_DIR / "AAS-41127-[41035.41127].pdb",
    DATA_DIR / "AAS-41130-[41038.41130].pdb",
    DATA_DIR / "AAS-41132-[41040.41132].pdb",
    DATA_DIR / "AAS-41139-[41047.41139].pdb",
    
]

# Proper descriptors

In [7]:
from AbXtract import AntibodyDescriptorCalculator

# Initialize calculator
calc = AntibodyDescriptorCalculator()

reduce not found at reduce
Missing external tools: reduce
Some analyses may not be available


--- 0.42188334465026855 seconds --- INITIALIZATION


# Sequence validity for numbering

In [8]:
df_fin = []
df_fin_heavy = []
df_fin_light = []

for HEAVY_SEQUENCE, LIGHT_SEQUENCE, PDB_FILE in zip(HEAVY_SEQUENCE_list, LIGHT_SEQUENCE_list, PDB_FILE_list):
    df_heavy_final, df_light_final, df_final = desc_Ab(HEAVY_SEQUENCE, LIGHT_SEQUENCE, PDB_FILE)
    df_fin.append(df_final)
    df_fin_heavy.append(df_heavy_final)
    df_fin_light.append(df_light_final)

--- 0.12493658065795898 seconds --- self.Calculate numbering
--- 0.10425829887390137 seconds --- self.config.calculate_liabilities
--- 0.07235956192016602 seconds --- self.config.calculate_bashour
--- 3.337860107421875e-06 seconds --- self.config.calculate_peptide
--- 7.3909759521484375e-06 seconds --- self.config.calculate_protpy
SeqID
Type
Heavy_Length
Light_Length
liabilities
count
total_possible
heavy_sequence
light_sequence
scheme
Heavy_Molecular Weight
Light_Molecular Weight
Heavy_Seq Length
Light_Seq Length
Heavy_Average Residue Weight
Light_Average Residue Weight
Heavy_pI
Light_pI
Heavy_Charges (all pH values)
Light_Charges (all pH values)
Heavy_Charge_pH_1
Light_Charge_pH_1
Heavy_Charge_pH_2
Light_Charge_pH_2
Heavy_Charge_pH_3
Light_Charge_pH_3
Heavy_Charge_pH_4
Light_Charge_pH_4
Heavy_Charge_pH_5
Light_Charge_pH_5
Heavy_Charge_pH_6
Light_Charge_pH_6
Heavy_Charge_pH_7
Light_Charge_pH_7
Heavy_Charge_pH_8
Light_Charge_pH_8
Heavy_Charge_pH_9
Light_Charge_pH_9
Heavy_Charge_pH_10
L

DSSP analysis failed: DSSP failed to produce an output


--- 0.14368605613708496 seconds --- Run DSSP
--- 1.0671792030334473 seconds --- Run Extended PROPKA


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f1ed14e9eb0>>
Traceback (most recent call last):
  File "/opt/conda/envs/abxtract/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
  File "_pydevd_bundle\\pydevd_cython.pyx", line 1697, in _pydevd_bundle.pydevd_cython.SafeCallWrapper.__call__
  File "_pydevd_bundle\\pydevd_cython.pyx", line 2017, in _pydevd_bundle.pydevd_cython.ThreadTracer.__call__
  File "/opt/conda/envs/abxtract/lib/python3.9/site-packages/debugpy/_vendored/pydevd/_pydev_bundle/pydev_is_thread_alive.py", line 20, in is_thread_alive
    def is_thread_alive(t):
KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f1ed14e9eb0>>
Traceback (most recent call last):
  File "/opt/conda/envs/abxtract/lib/python3.9/site-package

--- 48.27128529548645 seconds --- Run computeProper



KeyboardInterrupt


KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt

KeyboardInterrupt



KeyboardInterrupt























In [ ]:
df_test = pd.concat(df_fin, axis = 0)
df_mod = analysis_descriptors.prepare_object_descriptors(df_test)
df_mod

# Clean

In [ ]:
# Remove duplicate columns (keeps the first occurrence)
df_cleaned = df_mod.loc[:, ~df_mod.columns.duplicated(keep='first')]

# Verify the cleaning worked
print(f"Original shape: {df_mod.shape}")
print(f"Cleaned shape: {df_cleaned.shape}")
print(f"Removed {df_mod.shape[1] - df_cleaned.shape[1]} duplicate columns")

# Check if Heavy_molecular_weight still exists (should be only 1 now)
heavy_cols = [col for col in df_cleaned.columns if 'Heavy_molecular_weight' in col]
print(f"Heavy_molecular_weight columns remaining: {len(heavy_cols)}")

In [ ]:
out_ = []
for col in df_mod.columns.to_list():
    if "protpy" not in col:
        out_.append(col)

In [ ]:
df_mod = df_mod[out_]

In [ ]:

ids_Ab = ["AAS-41124-[41032.41124]",
          "AAS-41125-[41033.41125]",
          "AAS-41126-[41034.41126]",
          "AAS-41127-[41035.41127]",
          "AAS-41130-[41038.41130]",
          "AAS-41132-[41040.41132]",
          "AAS-41139-[41047.41139]"]

df_mod["Identifier"] = ids_Ab

In [ ]:
df_mod

In [ ]:
df_mod.style

In [ ]:
df_mod.to_csv("./CD33_desc.csv", index = None)

# Evolution pH

In [ ]:
ID_Fv = 0

In [ ]:
df_heavy_final = df_fin_heavy[ID_Fv]
df_light_final = df_fin_light[ID_Fv]
patterns = ["Light_Charges_pH_","Heavy_Charge_pH_","Free_Energy_kcal_mol_",
            "Protein_Charge_Unfolded_","Protein_Charge_Folded_pH_",
            "pI_Folded_pH_","pI_Unfolded_pH_"]
col_ph = sorted([col for col in df_test.columns 
                 if any(pattern in col for pattern in patterns)])
object_0_df = analysis_descriptors.reshape_dataframe_by_object(df_test[col_ph])[0]

In [ ]:
object_0_df

In [ ]:
fig = analysis_descriptors.plot_ph_profiles(object_0_df, object_id=0)
plt.show()


# Residue mapping

In [ ]:
fig = analysis_descriptors.plot_protein_properties(df_heavy_final, chain_type='heavy')
plt.show()


In [ ]:
fig_heavy = analysis_descriptors.plot_protein_properties(df_light_final, chain_type='light')
plt.show()


# Propka spe

In [ ]:
fig_propka_heavy = analysis_descriptors.plot_propka_properties(df_heavy_final, chain_type='heavy')
plt.show()


In [ ]:
fig_propka_light = analysis_descriptors.plot_propka_properties(df_light_final, chain_type='light')
plt.show()